# 03. AI Search 인덱싱 파이프라인 실행

이 노트북에서는:
1. AI Search 인덱스 스키마 및 설계 원칙을 이해합니다.
2. Blob Storage에 있는 크롤링 데이터 중 어떤 파일들이 인덱싱될 대상인지 확인합니다.
3. AI Search 인덱싱 파이프라인(skillset, datasource, indexer)을 생성하고 실행합니다.

권장 순서: `02 → 03 → 04`

## 1. 환경 설정

In [19]:
import os, sys
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

SEARCH_ENDPOINT = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
OPENAI_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']
STORAGE_NAME = os.environ.get('AZURE_STORAGE_ACCOUNT_NAME', '')
STORAGE_CONTAINER = os.environ.get('AZURE_STORAGE_CONTAINER_NAME', 'raw-documents')

print(f"Search Endpoint: {SEARCH_ENDPOINT}")
print(f"OpenAI Endpoint: {OPENAI_ENDPOINT}")
print(f"Storage         : {STORAGE_NAME}/{STORAGE_CONTAINER}")

Search Endpoint: https://search-ragi-dyn6dtfu.search.windows.net
OpenAI Endpoint: https://ais-ragi-dyn6dtfu.openai.azure.com/
Storage         : stragidyn6dtfun6/raw-documents


## 2. 인덱스 스키마 설계 — 법률 4개 인덱스

법률 데이터 특성에 맞게 인덱스별로 스키마를 분리 설계합니다.  
모든 인덱스는 **하이브리드 검색(키워드 + 벡터) + Semantic Ranker** 구조를 공유합니다.

### 공통 설계 원칙

| 구분 | 타입 | 설정 | 용도 |
|------|------|------|------|
| **Key 필드** | SimpleField | `key=True, filterable=True` | 문서 고유 ID |
| **메타데이터** | SearchableField | `filterable=True, sortable=True` | 날짜 범위 필터, 정렬 |
| **열거형 메타** | SearchableField | `filterable=True, facetable=True` | 심급·유형 패싯 UI |
| **다중값 메타** | SearchableField | `type=Collection(String), filterable=True` | 관련법령·주제어 정확 필터 |
| **본문 텍스트** | SearchableField | `analyzer_name="ko.microsoft"` | 한국어 키워드 검색 |
| **벡터 임베딩** | SearchField | `searchable=True, hidden=True` | 의미 기반 벡터 검색 |

**주요 결정**:
- **임베딩 비용 절약**: 전체 문서가 아닌 핵심 필드만 임베딩 (판시사항 + 판결요지 등)
- **`Collection(String)` 필터**: "민법,형법" 단일 텍스트로 저장 불가 → 배열로 변경
- **`hidden=True`**: 3072 float 벡터를 응답에서 제외 (네트워크 비용 감소)

In [20]:
from src.search.legal_indexes import PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX

# 4개 인덱스 정보
INDEX_META = {
    PREC_INDEX: {
        "name": "판례",
        "source": "prec",
        "blob_prefix": "prec/",
        "embedding_fields": ["판시사항", "판결요지"],
        "semantic_config": "prec-semantic",
    },
    CONST_INDEX: {
        "name": "헌법재판소 결정례",
        "source": "detc",
        "blob_prefix": "detc/",
        "embedding_fields": ["판시사항", "결정요지"],
        "semantic_config": "const-semantic",
    },
    INTERP_INDEX: {
        "name": "법제처 해석례",
        "source": "expc",
        "blob_prefix": "expc/",
        "embedding_fields": ["질의요지", "회답"],
        "semantic_config": "interp-semantic",
    },
    ADMIN_INDEX: {
        "name": "행정심판 재결례",
        "source": "admrul",
        "blob_prefix": "admrul/",
        "embedding_fields": ["주문", "이유요약"],
        "semantic_config": "admin-semantic",
    },
}

print("\n=== 4개 AI Search 인덱스 ===\n")
print(f"{'인덱스명':<30} {'한국어명':<20} {'Blob 경로':<15} {'임베딩 대상 필드'}")
print("-" * 85)
for idx_name, meta in INDEX_META.items():
    print(
        f"{idx_name:<30} "
        f"{meta['name']:<20} "
        f"{meta['blob_prefix']:<15} "
        f"{', '.join(meta['embedding_fields'])}"
    )


=== 4개 AI Search 인덱스 ===

인덱스명                           한국어명                 Blob 경로         임베딩 대상 필드
-------------------------------------------------------------------------------------
prec-court-index               판례                   prec/           판시사항, 판결요지
const-court-index              헌법재판소 결정례            detc/           판시사항, 결정요지
legis-interp-index             법제처 해석례              expc/           질의요지, 회답
admin-appeal-index             행정심판 재결례             admrul/         주문, 이유요약


## 3. 인덱싱될 Blob 파일 확인

Blob Storage에서 실제로 AI Search로 인덱싱될 4개 소스별 파일들을 확인합니다.

In [21]:
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

credential = DefaultAzureCredential()
blob_svc = BlobServiceClient(
    account_url=f"https://{STORAGE_NAME}.blob.core.windows.net",
    credential=credential,
)
container_client = blob_svc.get_container_client(STORAGE_CONTAINER)

# 각 소스별 Blob 파일 집계
print("\n=== Blob 파일 현황 (인덱싱될 데이터) ===\n")
total_files = 0
for idx_name, meta in INDEX_META.items():
    prefix = meta['blob_prefix']
    blobs = [b for b in container_client.list_blobs(name_starts_with=prefix) if b.name.endswith('.json')]
    total_files += len(blobs)
    
    # 날짜별 집계
    dates = sorted(set(b.name.split('/')[1] for b in blobs if b.name.count('/') >= 2), reverse=True)
    
    print(f"[{meta['name']}] ({prefix})")
    print(f"  전체: {len(blobs)}개 파일, {len(dates)}개 날짜")
    for d in dates[:3]:
        cnt = sum(1 for b in blobs if f"{prefix}{d}/" in b.name)
        print(f"    {d}: {cnt}개")
    print()

print(f"✅ 인덱싱될 총 파일: {total_files}개\n")


=== Blob 파일 현황 (인덱싱될 데이터) ===

[판례] (prec/)
  전체: 18841개 파일, 2개 날짜
    2026-05-13: 9573개
    2026-05-12: 9268개

[헌법재판소 결정례] (detc/)
  전체: 24985개 파일, 2개 날짜
    2026-05-13: 12004개
    2026-05-12: 12981개

[법제처 해석례] (expc/)
  전체: 8715개 파일, 1개 날짜
    2026-05-12: 8715개

[행정심판 재결례] (admrul/)
  전체: 6831개 파일, 2개 날짜
    2026-05-13: 3282개
    2026-05-12: 3549개

✅ 인덱싱될 총 파일: 59372개



## 4. AI Search 인덱싱 파이프라인 실행 (소스별 분리)

각 소스별로 인덱서를 개별 실행합니다. 한 소스가 실패해도 다른 소스에는 영향이 없으며, 시간/실패 건수도 분리해서 확인할 수 있습니다.

각 셀이 수행하는 작업:
1. 해당 소스 인덱스 생성 (스키마, Semantic Ranker 포함)
2. Skillset / DataSource / Indexer 생성
3. Indexer 즉시 실행 + 완료 대기 (최대 15분)


In [22]:
import subprocess
import time
from datetime import datetime


def run_indexing(source: str, label: str, timeout: int = 900):
    """단일 소스 인덱싱 파이프라인 실행 (생성 + Indexer 즉시 실행 + 완료 대기)"""
    print("\n" + "=" * 80)
    print(f"[{label}] AI Search 인덱싱 시작 (source={source})")
    print("=" * 80)
    print(f"시작 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80 + "\n")

    start = time.time()
    result = subprocess.run(
        [sys.executable, "scripts/setup_ai_search_pipeline.py", "--source", source, "--run"],
        cwd=os.path.abspath(".."),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=timeout,
    )
    elapsed = time.time() - start

    print(result.stdout)
    if result.stderr:
        print("\n[stderr]")
        print(result.stderr)

    print("\n" + "=" * 80)
    print(f"[{label}] 인덱싱 완료")
    print(f"종료 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"소요 시간: {elapsed:.1f}초 ({int(elapsed // 60)}분 {int(elapsed % 60)}초)")
    print(f"종료 코드: {result.returncode} ({'✅ 성공' if result.returncode == 0 else '❌ 실패'})")
    print("=" * 80)
    return result.returncode, elapsed


In [24]:
# (1/4) 판례 인덱싱
run_indexing("prec", "판례")



[판례] AI Search 인덱싱 시작 (source=prec)
시작 시간: 2026-05-13 16:14:44

AI Search 법률 인덱싱 파이프라인 설정
  Search    : https://search-ragi-dyn6dtfu.search.windows.net
  Container : processed-documents
  스케줄    : PT24H (start: 2026-01-01T06:00:00Z)
  대상      : prec

[사전 확인] AI Search Shared Private Link 승인 필요:
  az search shared-private-link-resource list \
      --service-name <search-name> --resource-group <rg>
  → status가 'Approved'인지 확인


────────────────────────────────────────────────────────────
  [prec] prec-court-index
────────────────────────────────────────────────────────────
[Index] prec-court-index
  - 기존 /indexes/prec-court-index 삭제
  ✓ Index 'prec-court-index' 생성 완료
[Skillset] prec-rag-skillset
  - 기존 /skillsets/prec-rag-skillset 삭제
  ✓ Skillset 'prec-rag-skillset' 생성 완료 (임베딩: /document/판결요지)
  - Indexer 'prec-blob-indexer' reset 완료
[DataSource] prec-blob-datasource
  - 기존 /datasources/prec-blob-datasource 삭제
  ✓ DataSource 'prec-blob-datasource' 생성 완료 (processed-documents/prec/)
[Ind

(0, 619.9778425693512)

In [25]:
# (2/4) 헌법재판소 결정례 인덱싱
run_indexing("detc", "헌법재판소 결정례")



[헌법재판소 결정례] AI Search 인덱싱 시작 (source=detc)
시작 시간: 2026-05-13 16:25:13

AI Search 법률 인덱싱 파이프라인 설정
  Search    : https://search-ragi-dyn6dtfu.search.windows.net
  Container : processed-documents
  스케줄    : PT24H (start: 2026-01-01T06:00:00Z)
  대상      : detc

[사전 확인] AI Search Shared Private Link 승인 필요:
  az search shared-private-link-resource list \
      --service-name <search-name> --resource-group <rg>
  → status가 'Approved'인지 확인


────────────────────────────────────────────────────────────
  [detc] const-court-index
────────────────────────────────────────────────────────────
[Index] const-court-index
  - 기존 /indexes/const-court-index 삭제
  ✓ Index 'const-court-index' 생성 완료
[Skillset] const-rag-skillset
  - 기존 /skillsets/const-rag-skillset 삭제
  ✓ Skillset 'const-rag-skillset' 생성 완료 (임베딩: /document/결정요지)
  - Indexer 'const-blob-indexer' reset 완료
[DataSource] const-blob-datasource
  - 기존 /datasources/const-blob-datasource 삭제
  ✓ DataSource 'const-blob-datasource' 생성 완료 (processed-doc

(0, 622.3036816120148)

In [26]:
# (3/4) 법제처 해석례 인덱싱
run_indexing("expc", "법제처 해석례")



[법제처 해석례] AI Search 인덱싱 시작 (source=expc)
시작 시간: 2026-05-13 16:35:41

AI Search 법률 인덱싱 파이프라인 설정
  Search    : https://search-ragi-dyn6dtfu.search.windows.net
  Container : processed-documents
  스케줄    : PT24H (start: 2026-01-01T06:00:00Z)
  대상      : expc

[사전 확인] AI Search Shared Private Link 승인 필요:
  az search shared-private-link-resource list \
      --service-name <search-name> --resource-group <rg>
  → status가 'Approved'인지 확인


────────────────────────────────────────────────────────────
  [expc] legis-interp-index
────────────────────────────────────────────────────────────
[Index] legis-interp-index
  - 기존 /indexes/legis-interp-index 삭제
  ✓ Index 'legis-interp-index' 생성 완료
[Skillset] interp-rag-skillset
  - 기존 /skillsets/interp-rag-skillset 삭제
  ✓ Skillset 'interp-rag-skillset' 생성 완료 (임베딩: /document/회답)
  - Indexer 'interp-blob-indexer' reset 완료
[DataSource] interp-blob-datasource
  - 기존 /datasources/interp-blob-datasource 삭제
  ✓ DataSource 'interp-blob-datasource' 생성 완료 (proces

(0, 286.23297810554504)

In [27]:
# (4/4) 행정심판 재결례 인덱싱
run_indexing("admrul", "행정심판 재결례")



[행정심판 재결례] AI Search 인덱싱 시작 (source=admrul)
시작 시간: 2026-05-13 16:40:33

AI Search 법률 인덱싱 파이프라인 설정
  Search    : https://search-ragi-dyn6dtfu.search.windows.net
  Container : processed-documents
  스케줄    : PT24H (start: 2026-01-01T06:00:00Z)
  대상      : admrul

[사전 확인] AI Search Shared Private Link 승인 필요:
  az search shared-private-link-resource list \
      --service-name <search-name> --resource-group <rg>
  → status가 'Approved'인지 확인


────────────────────────────────────────────────────────────
  [admrul] admin-appeal-index
────────────────────────────────────────────────────────────
[Index] admin-appeal-index
  - 기존 /indexes/admin-appeal-index 삭제
  ✓ Index 'admin-appeal-index' 생성 완료
[Skillset] admin-rag-skillset
  - 기존 /skillsets/admin-rag-skillset 삭제
  ✓ Skillset 'admin-rag-skillset' 생성 완료 (임베딩: /document/재결요지)
  - Indexer 'admin-blob-indexer' reset 완료
[DataSource] admin-blob-datasource
  - 기존 /datasources/admin-blob-datasource 삭제
  ✓ DataSource 'admin-blob-datasource' 생성 완료 (proc

(0, 287.7076642513275)

## 5. 인덱스 통계 확인

인덱싱 완료 후 4개 인덱스에 저장된 문서 개수를 확인합니다.

In [28]:
from src.search.legal_indexes import LegalIndexManager

mgr = LegalIndexManager(
    endpoint=SEARCH_ENDPOINT,
    admin_key=os.environ.get('AZURE_SEARCH_ADMIN_KEY'),
)

print("\n=== 인덱스 문서 개수 ===\n")
stats = mgr.get_all_stats()
total = 0
for s in stats:
    idx_name = s['index_name']
    meta = INDEX_META.get(idx_name, {})
    korean_name = meta.get('name', idx_name)
    count = s['document_count']
    total += count
    status = "✅" if count > 0 else "⚠️"
    print(f"{status} {korean_name:<25} ({idx_name:<30}): {count:,}건")

print(f"\n총합: {total:,}건")

if total > 0:
    print("\n✅ 인덱싱 완료! 다음 단계(04-search-and-query.ipynb)로 진행하세요.")
else:
    print("\n⚠️  아직 인덱싱된 문서가 없습니다. Blob 파일을 확인하세요.")


=== 인덱스 문서 개수 ===

✅ 판례                        (prec-court-index              ): 15,699건
✅ 헌법재판소 결정례                 (const-court-index             ): 23,908건
✅ 법제처 해석례                   (legis-interp-index            ): 8,123건
✅ 행정심판 재결례                  (admin-appeal-index            ): 6,588건

총합: 54,318건

✅ 인덱싱 완료! 다음 단계(04-search-and-query.ipynb)로 진행하세요.
